# Beyond BLAST Notebook

In [101]:
using Base.Threads
println("Threads available: ", Threads.nthreads())

Threads available: 8


In [102]:
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = 0;
using Pkg
Pkg.activate("bb_code") #the environment is in the blast_code folder. now i call it bb_code
Pkg.resolve()
Pkg.instantiate()

using Revise 
using HDF5, NPZ, DataInterpolations, Interpolations, FastChebInterp
using BenchmarkTools, FFTW, FastTransforms, Dates, TOML, Plots, Plots.Measures
using QuadGK, LaTeXStrings, Tullio, StaticArrays, LoopVectorization, LinearAlgebra
using Unitful, SpecialFunctions, DifferentialEquations, Cosmology, NumericalIntegration
using CSV, DataFrames, JSON, OrderedCollections
using ProgressMeter
ProgressMeter.ijulia_behavior(:clear)
;

  Activating project at `~/Desktop/cosmo/notebooks/bb_code`
  No Changes to `~/Desktop/cosmo/notebooks/bb_code/Project.toml`
  No Changes to `~/Desktop/cosmo/notebooks/bb_code/Manifest.toml`


#### Including the .jl modules

In [180]:
include("bb_code/src/bb.jl")
using .bb
;

#### Defining an output folder for each run
#### Setting run parameters, plot parameters, and grids in k

In [161]:
# #sets the paths for the output directories 
# paths = bb.setup_output_directories()
# bb.append_to_log(paths.output_dir, "=== Run started ===")
# #sets up the cosmology grid and returns the parameters
# grid_data = bb.setup_cosmology_grid() 
# #sets up the plotting theme and returns the parameters
# plot_theme = bb.setup_plot_theme(; paper = true) #if paper = true, dpi = 1000, else dpi = 300
# #returns the k grids for the calculations
# grids = bb.make_k_grids(grid_data.kmin, grid_data.kmax, 
#                            grid_data.Nk, grid_data.Nkp, grid_data.Nkpp; 
#                            sorting=true, output_dir = paths.output_dir) 
# #saves the run parameters in a txt file
# params_run = bb.save_run_config(
#     paths.output_dir, 
#     grid_data.N, 
#     grid_data.xmin, grid_data.xmax, 
#     grid_data.zmin, grid_data.zmax, 
#     grid_data.kmin, grid_data.kmax, 
#     grid_data.n_cheb, 
#     grid_data.ℓ, 
#     grid_data.Nk, grid_data.Nkp, grid_data.Nkpp,
#     grid_data.x, grid_data.z, 
#     grids.k_grid, grids.kp_grid, grids.kpp_grid, grids.sorting)
# ;


In [181]:
grid_data = bb.setup_cosmology_grid() 

(cosmo = Main.bb.FlatΛCDM{Float64}(-1.0, 0.0, 67.27, 0.3156, 0.0492, 0.6844, 2.12107e-9, 0.816, 0.0, 0.0, 0.9645), N = 32769, Nk = 150, Nkp = 150, Nkpp = 150, n_cheb = 100, z_b = [0.0, 0.09775171065493646, 0.19550342130987292, 0.29325513196480935, 0.39100684261974583, 0.4887585532746823, 0.5865102639296187, 0.6842619745845552, 0.7820136852394917, 0.8797653958944281  …  99.12023460410556, 99.21798631476051, 99.31573802541544, 99.41348973607037, 99.51124144672531, 99.60899315738025, 99.70674486803519, 99.80449657869012, 99.90224828934505, 100.0], x_b = [0.0, 425.3870057955954, 829.8103311819452, 1212.9800314518056, 1575.078943519494, 1916.659930228707, 2238.5416689139856, 2541.716140225368, 2827.2720667069316, 3096.3367900719286  …  12823.398298101381, 12824.160646636943, 12824.921882071527, 12825.682006075253, 12826.441020319799, 12827.198926478348, 12827.955726225538, 12828.711421237404, 12829.466013191313, 12830.219503765933], n5k_bins = Dict{String, Array{Float64}}("dNdz_cl" => [9.51

---

Computing $\tilde W$?
- reuse = false will compute $\tilde W$
- reuse = true will load an already computed $\tilde W$, using the mode below to compute a different one for each choice of the parameters
  - mode = "slow" will load the slowly-computed $\tilde W$
  - mode = "fast" will load the fastly-computed $\tilde W$ -> not here anymore, fast and slow are the same object
  - mode = "big" will load the huge $\tilde W$ -> now here anymore

In [122]:
reuse = false #if set to true, the code will load the W_tilde. 
default(fontfamily = "Computer Modern", titlefontfamily = "Computer Modern", legendfontfamily = "Computer Modern")

In [153]:
hist_k_log, hist_k, hist_kp, hist_kpp = bb.plot_k_grids(grids.k_grid, grids.kp_grid, grids.kpp_grid; 
                                                    output_dir = paths.output_dir, plot_style = plot_theme.shared_style)
;

In [108]:
println("k_grid goes from \n", grids.k_grid[1], " h/Mpc \nto \n", grids.k_grid[end], " h/Mpc")
println("sorting is ", grids.sorting)
println("kmin is ", grid_data.kmin, " h/Mpc", "\nkmax is ", grid_data.kmax, " h/Mpc")

k_grid goes from 
0.000357142857142857 h/Mpc 
to 
15.384615384615389 h/Mpc
sorting is true
kmin is 0.00035714285714285714 h/Mpc
kmax is 15.384615384615385 h/Mpc


### Galaxy clustering factor

$bias = b(z,z^2,z^3)$
comes from [this paper](https://arxiv.org/pdf/1807.10331)

Defining the kernel/window function for a galaxy probe: the window function is \
\
$W(z) = \frac{H(z) n(z) \chi(z)^2 b(z) D(z)}{c} $ \
\
where \
\
$n(z) = A (\frac{z}{z_0})^{\alpha} exp[{-(\frac{z}{z_0})^{\beta}}] $, \
\
with $A = \frac{1.5}{z_0}$, $\alpha = 2$ and $\beta = 1.5$ \
\
$b(z) = b_0 \sqrt{1+z} $ and $b_0 = 1$ \
\
$D(z) = \frac{D(z)^{unnorm}}{D(0)^{unnorm}}$, \
\
with $D(z)^{unnorm} = E(z) \int_z^{\infty} dz' \frac{1+z'}{E(z')^3} $ 

As for the ````gal_prefactor_W_cheb````, it is obtained interpolating each factor, ````bias````, ````growth````, ````nz_norm````, ````chi```` on the ````z````, and then obtaining the total interpolated product ````W_cheb````.

As for the ````c_cheb````, I compute this similarly to Blast: I define a ````plan```` object that takes the ````W_cheb```` as input, and then returns the ````cheb_coeff```` as the output of the functions ````fast_chebcoefs````

In [163]:
W_x, bias, growth, Hz, nz_norm = bb.compute_Wx(grid_data.x, grid_data.z, grid_data.cosmo; 
                                             output_dir=paths.output_dir, plot_style = plot_theme.shared_style);
W_cheb, x_cheb = bb.compute_Wcheb(grid_data.xmin, grid_data.xmax, grid_data.n_cheb, grid_data.z, grid_data.x, 
                                              bias, growth, Hz, nz_norm; 
                                              output_dir=paths.output_dir, sorting=grids.sorting);
c_cheb = bb.compute_c_cheb(W_cheb; output_dir=paths.output_dir, sorting=grids.sorting);

Is it true that

$W(\chi) \approx \sum_{n=0}^{N_{cheb}-1} c_n T_n(\chi)$ ?

In [182]:
W_x_on_cheb, rel_err_pct, errs, _, _, _, _, _, max_err =
    bb.analyze_W_cheb(grid_data, x_cheb, W_x, W_cheb, c_cheb, grids, bb, paths, plot_theme)
;

In [111]:
print(max_err)

0.08976866922898452

$W_{tilde} = \int dz W(z) j_l(k\chi(z)) j_l(k_1\chi(z))$

$\tilde W_{\ell}^g(k1,k) \approx \sum_{n=0}^{N_{cheb}-1} c_n \int_{z_{min}}^{z_{max}} dz T_n(\hat z) k_1 j_l(k\chi(z)) j_l(k_1\chi(z))$

Computing $\tilde W(k, k_1)$...

$\mathrm{N} = 2^{15}+1$, $\mathrm{N_k} = \mathrm{N_{kp}} = \mathrm{N_{kpp}} = 150$, $\mathrm{N_{cheb}} = 200$, $\mathrm{len}(ℓ) = 100$


In [112]:
W_tilde = zeros(grid_data.Nk, grid_data.Nkp, grid_data.n_cheb, length(grid_data.ℓ))
if reuse
    W_tilde = npzread("/Users/anvi/Desktop/cosmo/notebooks/out/runs/no_sorting_run_2026_07_20_102547/quantities/W_tilde.npy")
else
  p = Progress(length(grid_data.ℓ); desc = "Computing W_tilde")
  elapsed_time = zeros(length(grid_data.ℓ))
  println("Dimensions of W_tilde: ", size(W_tilde))
  for i in eachindex(grid_data.ℓ)
      t_0 = time()
      W_tilde[:, :, :, i] .= bb.compute_W(grid_data.ℓ[i], grid_data.zmin, grid_data.zmax,
                                          grid_data.Nk, grid_data.Nkp, grid_data.n_cheb, grid_data.N, 
                                          grids.k_grid, grids.kp_grid, grids.sorting)
      t_end = time()
      elapsed_time[i] = t_end - t_0
      next!(p; showvalues = [(:ℓ, grid_data.ℓ[i]), (:dt, round(elapsed_time[i], digits=2))])
  end
end
;

Dimensions of W_tilde: (150, 150, 100, 100)


LoadError: UndefVarError: `Dim_Integrated` not defined

Saving $\tilde W(k, k_1)$ and printing its dimensions...

In [ ]:
if reuse == false
  npzwrite(joinpath(paths.quantity_subdir, "W_tilde.npy"), W_tilde)
end
println("Size of W_tilde: (Nk, Nkp, Ncheb, Nℓ)")
println("Size of W_tilde: ", size(W_tilde))

$\tilde W(k,k_1)$ (like $\tilde W(k,k_2)$) represents the term: \
$\tilde W_{i,p,l}^{(\ell)} = \sum_{m=1}^{N_k} w_{k_m} T_{\ell}(k_m) j_{\ell}(\chi_i k_m) j_{\ell}(\chi_p k_m) $ \
it describes how much two shells at comoving distance $\chi_i$ and $\chi_p$ are correlated to the multipole $\ell$, weighted by the Chebyshev polynomial $T_{\ell}$ on the mode $k$.

Computing $\tilde W_{final}$ by contracting $\tilde W$ with $c_{cheb}$ and printing its dimensions...

In [ ]:
@tullio W_final_gal[il, ik, ikp] := W_tilde[ik, ikp, ic, il] * c_cheb[ic]

println("Size of W_final_gal: (Nℓ, Nk, Nkp)")
println("Size of W_final_gal: ", size(W_final_gal))
;

When sorting is set to true, the $k_{grid}$ has $k_{grid}[1] \approx k_{min}$, and $k_{grid}[end] \approx k_{max}$. \
When sorting is set to false, the $k_{grid}$ has $k_{grid}[1] \approx k_{max}$, and $k_{grid}[end] \approx k_{min}$.

In [ ]:
desired_ℓ_index = 1
bb.plot_heatmaps(W_final_gal, grids.k_grid, grids.kp_grid, plot_theme, paths; il = desired_ℓ_index)
;

In [ ]:
res = bb.plot_theory_Pk(grid_data, plot_theme; χ1 = 1000.0, χ2 = 1000.0)
display(res.p)

get_clencurt_grid produces the node of Clenshaw-Curtis mapped on [$k_{min}$, $k_{max}$]. \
get_clencurt_weights produces the corresponding quadrature weights scaled to the interval [-1,1]. \

In [ ]:
#Pk_grid = power_spectrum.(grids.k_grid, grid_data.xmin, grid_data.xmax)
Pk_grid = res.power_spectrum.(grids.k_grid, grid_data.xmin, grid_data.xmax)
Δk = diff(grids.k_grid)
w_trap = zeros(Float64, length(grids.k_grid))
w_trap[1] = 0.5 * Δk[1]
w_trap[2:end-1] = 0.5 * (Δk[1:end-1] .+ Δk[2:end])
w_trap[end] = 0.5 * Δk[end]
weight_gal = grids.k_grid.^2 .* Pk_grid .* w_trap
;

when plotting C(l) is doesn't matter that I define the plot with a grid in k which is decreasing (like k_grid). In the heatmap on the other hand, the axis must have ordered quantities.

In [ ]:
plot(grids.k_grid[2:end], Pk_grid[2:end], 
     xscale=:log10, 
     yscale=:log10, 
     title=L"$P(k)$", titlefontsize=20,
     xlabel=L"$k \; (h/\mathrm{Mpc})$", 
     ylabel=L"$P(k) \; ((\mathrm{Mpc}/h)^3)$", 
     labelfontsize=15, size=plot_theme.size_plot; plot_theme.shared_style...
     )

In [ ]:
plot(grids.k_grid[2:end], w_trap[2:end], label = L"weights", 
     xlabel = L"k \; (h/\mathrm{Mpc})", size=plot_theme.size_Cl; plot_theme.shared_style...
     )
plot!(grids.k_grid[2:end], grids.k_grid[2:end].^2, label = L"k^2",
     xlabel = L"k \; (h/\mathrm{Mpc})", size=plot_theme.size_Cl; plot_theme.shared_style...
     )
plot!(grids.k_grid[2:end], Pk_grid[2:end], label = L"P(k)", 
     xlabel = L"k \; (h/\mathrm{Mpc})", size=plot_theme.size_Cl; plot_theme.shared_style...
     )
plot!(grids.k_grid[2:end], weight_gal[2:end], 
     xscale=:log10, 
     yscale=:log10, 
     title=L"$k^2 P(k)$", titlefontsize=20,
     xlabel=L"$k \; (h/\mathrm{Mpc})$", 
     ylabel=L"$k^2 P(k) \; ((\mathrm{Mpc}/h))$", 
     label =L"$k^2 P(k) \; * \; \mathrm{w} $",
     labelfontsize=15, legendposition = :bottomright, size=plot_theme.size_plot; plot_theme.shared_style...
     )

In [ ]:
plot(grids.k_grid[2:end], weight_gal[2:end], 
     xscale=:log10, 
     yscale=:log10, 
     title=L"$k^2 P(k)$", titlefontsize=20,
     xlabel=L"$k \; (h/\mathrm{Mpc})$", 
     ylabel=L"$k^2 P(k) \; ((\mathrm{Mpc}/h))$", 
     label =L"$k^2 P(k) \; * \; \mathrm{w} $",
     labelfontsize=15, legendposition = :bottomright, size=plot_theme.size_plot; plot_theme.shared_style...
     )

In [ ]:
abstract type AbstractProbe end
struct Galaxy <: AbstractProbe end
struct Shear <: AbstractProbe end
factorial_frac(ℓ) = (ℓ + 2.0) * (ℓ + 1.0) * ℓ * (ℓ - 1.0)
get_ell_prefactor(::Galaxy, ::Galaxy, ℓ) = @. (2 / π) * ones(length(ℓ))
get_ell_prefactor(::Galaxy, ::Shear,  ℓ) = @. (2 / π) * sqrt(factorial_frac(ℓ))
get_ell_prefactor(::Shear,  ::Galaxy, ℓ) = @. (2 / π) * sqrt(factorial_frac(ℓ))
get_ell_prefactor(::Shear,  ::Shear,  ℓ) = @. (2 / π) * factorial_frac(ℓ)
pref_gg = get_ell_prefactor(Galaxy(), Galaxy(), grid_data.ℓ)
pref_gg = reduce(vcat, pref_gg)
pref_gs = get_ell_prefactor(Galaxy(), Shear(), grid_data.ℓ)
pref_gs = reduce(vcat, pref_gs)
pref_gg = reduce(vcat, pref_gg)
pref_ss = get_ell_prefactor(Shear(), Shear(), grid_data.ℓ)
pref_ss = reduce(vcat, pref_ss);

In [ ]:
println("SIZES")
println("weight_gal -> ", size(weight_gal))
println("W_final_gal -> ", size(W_final_gal))
println("pref_gg -> ", size(pref_gg))

In [ ]:
S_lkk_gg = zeros(Float64, size(W_final_gal, 3), size(W_final_gal, 3), length(grid_data.ℓ))
@tullio S_lkk_gg[kp, kpp, li] = pref_gg[li] * weight_gal[k] * W_final_gal[li, k, kp] * W_final_gal[li, k, kpp]
npzwrite(joinpath(paths.quantity_subdir, "Sl/S_lkk_gg.npy"), S_lkk_gg)
println("Size of S_l (kp, kpp) (gal-gal): \n(grid_data.Nk, grid_data.Nkp, NL) -> ", size(S_lkk_gg))
;

---

In [ ]:
bb.plot_Sl_kp_kpp(grids, S_lkk_gg, grid_data, paths, plot_theme; i=150, j = 150, save_fig=true, showfig=false)
bb.plot_Sl_fixed_l_kpp(grids, S_lkk_gg, grid_data, paths, plot_theme; il=1, ikpp=5)
bb.plot_Sl_fixed_l_varying_k(grids, S_lkk_gg, grid_data, paths, plot_theme; il=1, step=20, normalize=true)
bb.plot_Sl_fixed_l_varying_k(grids, S_lkk_gg, grid_data, paths, plot_theme; il=1, step=20, normalize=false)
bb.animate_Sl_fixed_l_varying_k(grids, S_lkk_gg, grid_data, paths, plot_theme; il=1, step=20, fps=2)
bb.plot_Sl_fixed_kpp_varying_l(grids, S_lkk_gg, grid_data, paths, plot_theme; ipp=4, step=20, normalize=false)
bb.animate_Sl_fixed_kpp_varying_l(grids, S_lkk_gg, grid_data, paths, plot_theme; ipp=140, step=50, fps=1)
bb.plot_l_lplus1_Sl(grids, S_lkk_gg, grid_data, ip = 150, ipp = 150, paths, plot_theme)
bb.plot_Sl_diagonal_ell_evolution(grids, S_lkk_gg, grid_data, paths, plot_theme)
;